# Polynomial & Regularized Regression
**Chapter 4 — Hands-On ML**

Covers: Polynomial features, learning curves, Ridge, Lasso, Elastic Net, Early Stopping.

Dataset: Synthetic — `y = 0.5x² + x + 2 + noise`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, SGDRegressor, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.base import clone
import warnings
warnings.filterwarnings('ignore')

## 1. Dataset

In [ ]:
np.random.seed(42)
m = 180
X = 6 * np.random.rand(m, 1) - 3
y = 0.5 * X**2 + X + 2 + np.random.rand(m, 1)

plt.scatter(X, y, alpha=0.5)
plt.xlabel('X'); plt.ylabel('y'); plt.title('Synthetic Quadratic Dataset')
plt.show()

## 2. Polynomial Features

In [ ]:
poly_features = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly_features.fit_transform(X)
print('Original feature:', X[0])
print('Polynomial features (x, x²):', X_poly[0])

In [ ]:
lin_reg = LinearRegression()
lin_reg.fit(X_poly, y)
print('Intercept:', lin_reg.intercept_)
print('Coefficients (x, x²):', lin_reg.coef_)

## 3. Learning Curves

Diagnose underfitting vs overfitting by plotting train/val error as training set size grows.

In [ ]:
def plot_learning_curves(model, X, y, title='Learning Curves'):
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
    train_errors, val_errors = [], []
    for m in range(1, len(X_train)):
        model.fit(X_train[:m], y_train[:m])
        y_train_pred = model.predict(X_train[:m])
        y_val_pred = model.predict(X_val)
        train_errors.append(mean_squared_error(y_train[:m], y_train_pred))
        val_errors.append(mean_squared_error(y_val, y_val_pred))
    plt.plot(np.sqrt(train_errors), 'r-+', linewidth=2, label='Train')
    plt.plot(np.sqrt(val_errors), 'b-', linewidth=2, label='Validation')
    plt.xlabel('Training Set Size'); plt.ylabel('RMSE')
    plt.title(title); plt.legend(); plt.show()

In [ ]:
plot_learning_curves(LinearRegression(), X, y, title='Linear Model — Underfitting')

In [ ]:
poly_pipeline = Pipeline([
    ('poly_features', PolynomialFeatures(degree=10, include_bias=False)),
    ('lin_reg', LinearRegression()),
])
plot_learning_curves(poly_pipeline, X, y, title='Degree-10 Polynomial — Overfitting')

## 4. Regularization

| Method | Penalty | Effect |
|--------|---------|--------|
| Ridge | L2 (sum of squares) | Shrinks all weights |
| Lasso | L1 (sum of absolutes) | Drives weights to 0 |
| Elastic Net | L1 + L2 | Blend of both |

### 4a. Ridge Regression

In [ ]:
ridge_reg = Ridge(alpha=1, solver='cholesky')
ridge_reg.fit(X, y)
print('Ridge prediction at x=1.5:', ridge_reg.predict([[1.5]]))

In [ ]:
sgd_ridge = SGDRegressor(penalty='l2', eta0=0.01, random_state=42)
sgd_ridge.fit(X, y.ravel())
print('SGD Ridge prediction at x=1.5:', sgd_ridge.predict([[1.5]]))

### 4b. Lasso Regression

In [ ]:
lasso_reg = Lasso(alpha=0.1)
lasso_reg.fit(X, y)
print('Lasso prediction at x=1.5:', lasso_reg.predict([[1.5]]))

### 4c. Elastic Net

In [ ]:
elastic_net = ElasticNet(alpha=0.1, l1_ratio=0.5)
elastic_net.fit(X, y)
print('ElasticNet prediction at x=1.5:', elastic_net.predict([[1.5]]))

## 5. Early Stopping

Halt training when validation error stops decreasing — prevents overfitting without explicit regularization.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

poly_scaler = Pipeline([
    ('poly_features', PolynomialFeatures(degree=90, include_bias=False)),
])
X_train_poly = poly_scaler.fit_transform(X_train)
X_val_poly = poly_scaler.transform(X_val)

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_poly)
X_val_scaled = scaler.transform(X_val_poly)

sgd_reg = SGDRegressor(max_iter=1, warm_start=True, penalty=None, learning_rate='constant', eta0=0.0005, random_state=42)

minimum_val_error = float('inf')
best_epoch = None
best_model = None

for epoch in range(1000):
    sgd_reg.fit(X_train_scaled, y_train.ravel())
    y_val_pred = sgd_reg.predict(X_val_scaled)
    val_error = mean_squared_error(y_val, y_val_pred)
    if val_error < minimum_val_error:
        minimum_val_error = val_error
        best_epoch = epoch
        best_model = clone(sgd_reg)

print(f'Best epoch: {best_epoch}, Val RMSE: {np.sqrt(minimum_val_error):.4f}')